# Databricks MDM Accelerator
This notebook is designed for Databricks Unity Catalog Bronze tables.

In [ ]:
from __future__ import annotations

import json
import logging
import re
from datetime import datetime
from typing import Any, Dict, List

from rapidfuzz import fuzz
from pyspark.sql import DataFrame, SparkSession

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("databricks_mdm")

CATALOG = "mdm_demo"
SCHEMA = "bronze"

spark = SparkSession.getActiveSession()


def normalize_name(value: Any) -> str | None:
    if value is None:
        return None
    cleaned = re.sub(r"\s+", " ", str(value).strip())
    return cleaned.upper() if cleaned else None


def normalize_email(value: Any) -> str | None:
    if value is None:
        return None
    cleaned = str(value).strip().lower()
    return cleaned if cleaned else None


def normalize_phone(value: Any) -> str | None:
    if value is None:
        return None
    digits = re.sub(r"\D+", "", str(value).strip())
    if not digits:
        return None
    if len(digits) > 10 and digits.startswith("91"):
        digits = digits[2:]
    elif len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]
    return digits


def classify_column(column_name: str) -> str:
    config = {
        "EMAIL": ["email", "mail", "email_id", "work_email"],
        "PHONE": ["phone", "mobile", "mobile_no", "cell"],
        "NAME": ["name", "full_name", "cust_name", "customer_name"],
        "ADDRESS": ["address", "addr", "street_address"],
    }
    norm = re.sub(r"[^a-z0-9_]", "", str(column_name).lower().replace("-", "_").replace(" ", "_"))
    for category, aliases in config.items():
        for alias in aliases:
            alias_norm = re.sub(r"[^a-z0-9_]", "", str(alias).lower().replace("-", "_").replace(" ", "_"))
            if norm == alias_norm or alias_norm in norm or norm in alias_norm:
                return category
    return "UNKNOWN"


def standardize_value(value: Any, classification: str) -> Any:
    mapping = {"NAME": normalize_name, "EMAIL": normalize_email, "PHONE": normalize_phone}
    fn = mapping.get(classification)
    if fn:
        return fn(value)
    return str(value).strip() if value is not None else None


def build_mdm_temp(df: DataFrame, source_system: str, entity_type: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for record in df.toPandas().to_dict(orient="records"):
        record_id = str(record.get("record_id") or record.get("id") or f"{source_system}_{hash(str(record)) % 1000000}")
        for key, value in record.items():
            classification = classify_column(key)
            if classification == "UNKNOWN":
                continue
            standardized = standardize_value(value, classification)
            if standardized is None:
                continue
            rows.append({
                "record_id": record_id,
                "source_system": source_system,
                "entity_type": entity_type,
                "attribute_name": classification,
                "attribute_value": standardized,
                "load_timestamp": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")
            })
    return rows


def exact_match(left: Any, right: Any) -> bool:
    return str(left).strip() == str(right).strip() if left is not None and right is not None else False


def fuzzy_match(left: Any, right: Any) -> float:
    if left is None or right is None:
        return 0.0
    return float(fuzz.ratio(str(left).strip(), str(right).strip()))


def build_match_candidates(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    candidates: List[Dict[str, Any]] = []
    for idx, left in enumerate(records):
        for right in records[idx + 1:]:
            score = 0.0
            for key in set(left.keys()) | set(right.keys()):
                if key in {"record_id", "source_system", "entity_type", "load_timestamp"}:
                    continue
                if left.get(key) is None or right.get(key) is None:
                    continue
                if key in {"EMAIL", "PHONE"}:
                    score += 100.0 if exact_match(left.get(key), right.get(key)) else 0.0
                else:
                    score += fuzzy_match(left.get(key), right.get(key))
            if score > 0:
                candidates.append({
                    "record_id_1": left.get("record_id"),
                    "record_id_2": right.get("record_id"),
                    "match_score": round(score, 2),
                    "match_status": "MATCH"
                })
    return candidates


source_tables = ["bronze.crm_customers", "bronze.banking_customers", "bronze.creditcard_customers"]
all_rows = []
for table_name in source_tables:
    df = spark.table(table_name)
    logger.info("Reading %s", table_name)
    all_rows.extend(build_mdm_temp(df, source_system=table_name.split(".")[-1], entity_type="CUSTOMER"))

mdm_temp_df = spark.createDataFrame(all_rows)
mdm_temp_df.write.mode("overwrite").saveAsTable("mdm_demo.bronze.mdm_temp")

match_rows = build_match_candidates(all_rows)
match_df = spark.createDataFrame(match_rows)
match_df.write.mode("overwrite").saveAsTable("mdm_demo.bronze.match_candidates")

print("mdm_temp and match_candidates built successfully")